In [13]:
### ---------------------------------
# Import
### ---------------------------------

import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import torch
from torch.utils.data import Dataset
from tqdm import tqdm

import albumentations as trans
from albumentations.pytorch import ToTensorV2


In [15]:


filename = "D:\\GPUAccess\\Anton\\ofdma-spectrum-anomalies-simulation\\1\\dataset.h5"

with h5py.File(filename, "r") as f:
    def print_structure(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"[Group]   {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"[Dataset] {name}")
            print(f"  Shape: {obj.shape}")
            print(f"  Dtype: {obj.dtype}")
            print(f"  Attributes: {dict(obj.attrs)}")

    print("File attributes:")
    print(dict(f.attrs))
    print("\nContents:")
    f.visititems(print_structure)

File attributes:
{}

Contents:
[Dataset] resource_allocations
  Shape: (20000, 1320, 70)
  Dtype: uint8
  Attributes: {}
[Dataset] spectrograms
  Shape: (20000, 21, 1320, 70)
  Dtype: uint8
  Attributes: {}


In [11]:
with h5py.File(filename, "r") as f:
    specs = f["spectrograms"]

    # First sample
    x = specs[0]

In [5]:
label_path = "D:\\GPUAccess\\Anton\\ofdma-spectrum-anomalies-simulation\\1\\labels.csv"
df = pd.read_csv(label_path)

C:\Users\Mohammadhadi.Salehi\AppData\Local\Temp\3\ipykernel_21644\535415547.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(label_path)


In [7]:
df

,jammer_type,jammer_power,jammer_location,num_legitimate_transmitters,jammer_occupancy,snr_by_su_0,snr_by_su_1,snr_by_su_2,snr_by_su_3,snr_by_su_4,...,db_contrast_local_by_su_13,db_contrast_local_by_su_14,db_contrast_local_by_su_15,db_contrast_local_by_su_16,db_contrast_local_by_su_17,db_contrast_local_by_su_18,db_contrast_local_by_su_19,db_contrast_local_by_su_20,split_supervised,split_unsupervised
0,no jammer,NaN,NaN,4,NaN,20.852594,27.941341,31.742985,25.877398,23.051765,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train,train
1,no jammer,NaN,NaN,10,NaN,29.853984,27.970865,40.252619,40.476766,34.827978,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train,train
2,no jammer,NaN,NaN,4,NaN,18.486117,25.537614,27.237836,25.578269,22.191764,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train,train
3,no jammer,NaN,NaN,10,NaN,65.245748,25.510654,23.420590,35.911450,42.124163,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,train,train
4,no jammer,NaN,NaN,5,NaN,20.868308,27.209988,20.346149,18.621174,17.226635,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,test,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,random_hop,10.0,[38.02333456 31.42762574 1.5 ],4,0.000584,5.815394,23.553986,27.500617,14.252383,16.366087,...,13.253703,19.871164,14.009614,12.935816,14.010037,12.631473,18.801569,23.463024,NaN,NaN
19996,random_hop,0.0,[ 0.18364577 11.64398699 1.5 ],9,0.000649,32.764477,34.635569,30.818354,28.606014,19.057759,...,36.446732,20.793976,29.962955,32.754392,27.055347,32.430233,27.150526,30.446941,test,test
19997,random_hop,10.0,[ 1.89733218 20.83445998 1.5 ],7,0.000346,19.712968,19.975534,23.429561,21.293186,19.193447,...,33.262331,22.387506,24.682637,21.100052,13.922958,9.058154,28.617136,22.375129,NaN,NaN
19998,random_hop,25.0,[ 8.03110569 19.82568114 1.5 ],6,0.000519,35.076485,44.290234,29.582145,11.268392,15.641800,...,71.334600,60.619799,59.595408,71.487266,59.544041,68.379797,67.730039,71.094180,test,test


## Data Loading Function

In [ ]:
def aggregate_subcarriers(spectrogram, num_sc_per_rb) -> np.ndarray:
    """
    Aggregate spectrogram subcarriers into resource blocks.

    Args:
        spectrogram (np.ndarray):
            Input spectrogram with shape (num_subcarriers, num_time_steps).

        num_sc_per_rb (int):
            Number of subcarriers per resource block.

    Returns:
        np.ndarray:
            Aggregated spectrogram.
    """
    num_rows, num_cols = spectrogram.shape
    num_aggregated_rows = num_rows // num_sc_per_rb
    aggregated_spectrogram = np.zeros((num_aggregated_rows, num_cols))

    for i in range(num_aggregated_rows):
        start_row = i * num_sc_per_rb
        end_row = start_row + num_sc_per_rb
        aggregated_spectrogram[i, :] = 10*np.log10(np.sum(10**(spectrogram[start_row:end_row, :] / 10), axis=0))

    return aggregated_spectrogram


base_transform = trans.Compose(
    [
        ToTensorV2(),
    ]
)

In [ ]:
# Data Classes ------------------------------------------------

class SpectralImagesUnsupervised(Dataset):
    """
    Dataset class for loading spectrogram image patches for an Unsupervised Learning task.
    """

    def __init__(self, filename: str, transform, dataframe, load_dt_images=False):

        if load_dt_images:
            raise NotImplementedError("Loading dt_images is currently not implemented.")

        self.transform = transform  
        self.filename = filename  
        self.id = []  
        self.frames = []  
        self.jammer = []  # list to hold jam type labels (string initially, mapped later)
        self.num_transmitters = []  # list to hold number of legitimate transmitters for each example
        self.jammer_power = []  # list to hold jammer power value for each example
        self.label = []

        with h5py.File(self.filename, "r") as f:
            specs = f["spectrograms"]

        # Iterate over remaining examples and cache per-sensing unit images
        for i, sample in tqdm(enumerate(dataframe), desc="Caching images"):
            for su in range(21):  # there are 21 sensing unit per sample (0..20)
                su_id = f'{i:05d}-{su:02d}'  # format example and sensing unit into an id string
                self.id.append(su_id)  

                img = specs[i, su, :, :]
                img = aggregate_subcarriers(img, num_sc_per_rb=12)

                # If a transform is provided, apply it (albumentations expects keyword 'image')
                if self.transform is not None:
                    img_transformed = self.transform(image=img)
                    self.frames.append(img_transformed["image"])
                else:
                    self.frames.append(img)

                self.jammer.append(sample['jammer_type'])
                self.num_transmitters.append(sample['num_legitimate_transmitters'])
                self.jammer_power.append(sample['jammer_power'])
                self.label.append(sample['split_unsupervised'])

        # Print available label distribution (strings) for debugging
        print("Sample Labels: \n", np.unique(self.jammer, return_counts=True))

        # Map jammer labels to integer classes; unknown -> -1
        jammer_map = {
            "no jammer": 0,
            "barrage": 1,
            "deceptive": 2,
            "pilot": 3,
            "sweep": 4,
            "random_hop": 5,
        }

        # Replace string labels with integer codes
        self.jammer = [jammer_map.get(jam, -1) for jam in self.jammer]

        # Print integer class distribution for debugging
        print("Jammer Classes: \n", np.unique(self.jammer, return_counts=True))

    def __len__(self) -> int:
        # Return number of cached frames
        return len(self.frames)

    def __getitem__(self, index):
        # Return a tuple for training/evaluation: (id, image, jammer_class, num_transmitters, jammer_power)
        return (
            self.id[index],
            self.frames[index],
            float(self.jammer[index]),
            float(self.num_transmitters[index]),
            float(self.jammer_power[index]),
        )


In [ ]:
train_samples = df[df['split_unsupervised'] == 'train']

trainset = SpectralImagesUnsupervised(
    filename,
    transform=base_transform,
    dataframe=train_samples,
    load_dt_images=False,
)
